# Build a minimal tool-use agent loop from scratch

Most "agent frameworks" are, at their core, a single small loop. Once you can
write that loop yourself, the frameworks stop being magic — they become a
convenience you reach for *when you want to*, not a black box you depend on.

In this recipe we build that loop from scratch with the `anthropic` SDK. No
framework, no abstractions — about 30 lines of Python. By the end you'll be able
to read any agent library and recognize exactly what it's doing under the hood.

**What we'll cover:**

1. What an "agent loop" actually *is* (the core idea in one paragraph)
2. Defining two simple tools with an `input_schema`
3. Writing the loop: call the model, run the tools it asks for, feed the results
   back, repeat
4. Handling **multiple tool calls in a single turn**
5. A worked example: *"What is (12 * 9) + 7, and what time is it?"*

This notebook only uses tools defined locally in Python (often called
*client-side* tools), so it runs anywhere the SDK runs.

## The core idea

When you give Claude a set of tools, a single API call no longer always returns a
final answer. Instead, the model can **pause and ask you to run a tool** on its
behalf. It can't execute code itself — *you* are the runtime. So the interaction
becomes a back-and-forth:

```text
  ┌──────────────────────────────────────────────────────────┐
  │  1. You send the conversation + the list of tools         │
  │  2. Claude replies. Two possibilities:                    │
  │       • stop_reason == "end_turn"  → it's done. Stop.     │
  │       • stop_reason == "tool_use"  → it wants a tool run  │
  │  3. You run each requested tool and collect the results   │
  │  4. You append those results to the conversation          │
  │  5. Go back to step 1                                     │
  └──────────────────────────────────────────────────────────┘
```

That loop — *call, run tools, feed results back, repeat until the model stops* —
**is** the agent. Everything else (planning, memory, multi-agent orchestration)
is built on top of this primitive. Let's implement it.

## Step 1: Set up the client

Install the SDK and create a client. The client reads your key from the
`ANTHROPIC_API_KEY` environment variable, so nothing secret is ever written into
the notebook.

In [ ]:
%pip install anthropic

In [ ]:
from anthropic import Anthropic

client = Anthropic()  # reads ANTHROPIC_API_KEY from the environment
MODEL_NAME = "claude-sonnet-4-6"

## Step 2: Define two tools

A tool, from the model's point of view, is just a **name**, a **description**,
and an **`input_schema`** (a [JSON Schema](https://json-schema.org/) describing
the arguments). Claude reads these and decides when to call them and with what
arguments — so write the description as if it were documentation for a teammate.

We'll define two small tools:

- `calculator` — evaluates an arithmetic expression
- `get_time` — returns the current time, optionally for a given timezone

Each tool is split into two halves that are easy to mix up, so name them
deliberately:

- the **schema** (the dict below) is what we send *to* the model, and
- the **Python function** is what *we* run when the model asks for it.

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo

# --- The Python functions we actually run ---------------------------------


def calculator(expression: str) -> str:
    """Evaluate a basic arithmetic expression and return the result as a string.

    NOTE: eval() is used here only to keep the example short. Never call eval()
    on untrusted input in production — use a real expression parser (e.g. the
    `ast` module or a math-parsing library) instead.
    """
    allowed = set("0123456789+-*/(). ")
    if not set(expression) <= allowed:
        return f"Error: unsupported characters in expression: {expression!r}"
    try:
        return str(eval(expression))  # noqa: S307 - demo only, input is filtered above
    except Exception as exc:  # noqa: BLE001 - surface any math error back to the model
        return f"Error: could not evaluate {expression!r} ({exc})"


def get_time(timezone: str = "UTC") -> str:
    """Return the current time in the given IANA timezone (e.g. 'Europe/Paris')."""
    try:
        now = datetime.now(ZoneInfo(timezone))
    except Exception:  # noqa: BLE001 - unknown timezone -> fall back to UTC
        return f"Error: unknown timezone {timezone!r}. Try 'UTC' or 'Europe/Paris'."
    return now.strftime("%Y-%m-%d %H:%M:%S %Z")


# --- The schemas we send to the model -------------------------------------

tools = [
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression and return the result.",
        "input_schema": {
            "type": "object",
            "properties": {
                "expression": {
                    "type": "string",
                    "description": "The arithmetic expression to evaluate, e.g. '(12 * 9) + 7'.",
                }
            },
            "required": ["expression"],
        },
    },
    {
        "name": "get_time",
        "description": "Get the current date and time, optionally for a specific timezone.",
        "input_schema": {
            "type": "object",
            "properties": {
                "timezone": {
                    "type": "string",
                    "description": (
                        "An IANA timezone name like 'UTC' or 'Europe/Paris'. Defaults to UTC."
                    ),
                }
            },
            "required": [],
        },
    },
]

We also need one helper to map a tool **name** back to the right Python function.
This is the single point where the model's request meets our code:

In [ ]:
def run_tool(name: str, tool_input: dict) -> str:
    """Dispatch a tool call coming from the model to the matching Python function."""
    if name == "calculator":
        return calculator(tool_input["expression"])
    if name == "get_time":
        return get_time(tool_input.get("timezone", "UTC"))
    # Returning an error string (rather than raising) lets the model see what went
    # wrong and recover on the next turn.
    return f"Error: unknown tool {name!r}"

## Step 3: The agent loop

Here is the whole thing. Read it once top-to-bottom — it's intentionally short,
because the point of this notebook is that the loop *is* short.

The shape of a single API response matters, so it's worth naming the pieces:

- `response.content` is a **list of blocks**. A turn can mix `text` blocks
  (things Claude says) and `tool_use` blocks (tools it wants run) — sometimes
  several of each.
- `response.stop_reason` tells us *why* the model stopped. We only care about
  two values: `"tool_use"` (it paused to call tools) and `"end_turn"` (it's
  finished).

So the loop is: while the model keeps asking for tools, run **every** tool it
requested this turn, hand back **all** the results in a single `user` message,
and call again. The moment `stop_reason` is no longer `"tool_use"`, we're done.

In [ ]:
def run_agent_loop(user_message: str, max_turns: int = 10) -> str:
    """Run the tool-use loop until the model produces a final answer.

    Returns the model's final text response.
    """
    messages = [{"role": "user", "content": user_message}]

    for _ in range(max_turns):
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=1024,
            tools=tools,
            messages=messages,
        )

        # The assistant turn (text and/or tool_use blocks) goes into the history
        # verbatim, so the model sees its own previous request on the next call.
        messages.append({"role": "assistant", "content": response.content})

        # No tool requested -> the model is done. Return its text.
        if response.stop_reason != "tool_use":
            return "".join(block.text for block in response.content if block.type == "text")

        # Otherwise, run every tool the model asked for in THIS turn and collect
        # one tool_result per request. Each result must reference its tool_use_id.
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                result = run_tool(block.name, block.input)
                print(f"  -> {block.name}({block.input}) = {result}")
                tool_results.append(
                    {
                        "type": "tool_result",
                        "tool_use_id": block.id,
                        "content": result,
                    }
                )

        # Send all results back as a single user turn, then loop to call again.
        messages.append({"role": "user", "content": tool_results})

    return "Stopped: reached max_turns without a final answer."

That's the entire agent. A few details worth calling out, because they're the
parts people most often get wrong the first time:

- **Append the assistant turn unchanged.** We push `response.content` (the raw
  block list) straight back into `messages`. The model needs to see its own
  `tool_use` request to make sense of the `tool_result` we send next.
- **One `tool_result` per `tool_use`, matched by `tool_use_id`.** This id is how
  the model pairs each result with the request that produced it.
- **All results go in a *single* `user` message.** Even when the model asked for
  three tools, you reply with one user turn containing three `tool_result`
  blocks — not three separate messages.
- **`max_turns` is a safety belt.** Without a ceiling, a misbehaving loop could
  call the API forever. Bounding the turns is cheap insurance.

## Step 4: Multiple tools in one turn

Notice that Step 3 already handles the multi-tool case — and that's the whole
reason we looped over `response.content` instead of grabbing the first
`tool_use` block.

When a request needs two independent things ("calculate X **and** tell me the
time"), Claude can emit **two `tool_use` blocks in the same turn**. Our loop
runs both, appends both results to one `user` message, and calls again. The
model then weaves both results into a single answer. A naïve implementation that
only handled `response.content[1]` would silently drop the second tool — so
iterating over the blocks is the thing that makes this correct.

## Step 5: Try it out

Let's run a question that needs **both** tools at once. Watch the `->` lines: you
should see `calculator` and `get_time` both get called before the final answer
comes back.

In [ ]:
question = "What is (12 * 9) + 7, and what time is it in Paris?"

print(f"Q: {question}\n")
answer = run_agent_loop(question)
print(f"\nA: {answer}")

And a single-tool question, to confirm the loop also handles the simple case —
one tool call, one result, then `end_turn`:

In [ ]:
print(run_agent_loop("What is 1984 * 2099?"))

## Recap & where to go next

You just built a complete tool-use agent in about 30 lines. The whole thing is
one loop:

> **Call the model. If it asked for tools, run them, append the results, and call
> again. Otherwise, you're done.**

Everything more advanced is a variation on this core:

- **More tools** — add entries to `tools` and branches to `run_tool`. The loop
  doesn't change.
- **Streaming** — swap `client.messages.create(...)` for
  `client.messages.stream(...)` to show progress token-by-token.
- **Parallel execution** — when the model requests several tools at once, run
  them concurrently (e.g. with `asyncio` or a thread pool) instead of
  sequentially in the `for` loop.
- **Real side effects** — point your tools at databases, HTTP APIs, or a shell.
  The agent's usefulness comes entirely from what its tools can *do*.

To go deeper, see the other tool-use recipes in this directory — for example
[`parallel_tools.ipynb`](parallel_tools.ipynb) for encouraging parallel calls,
[`tool_choice.ipynb`](tool_choice.ipynb) for forcing or disabling tool use, and
[`customer_service_agent.ipynb`](customer_service_agent.ipynb) for a fuller
client-side-tools example. The official
[tool use documentation](https://docs.claude.com/en/docs/build-with-claude/tool-use)
covers the API in full.